## *Fashion MNIST* konvoliucinis neuroninis tinklas

### Bibliotekų improtavimas

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import datasets, layers, models, activations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import copy
import os
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def create_dir(path):
  if not os.path.exists(path):
    os.makedirs(path)

# dir for all exports
create_dir("/content/drive/MyDrive/conv")

def prefix_dir(title_ext):
  return "/content/drive/MyDrive/conv/" + title_ext

In [ ]:
# TODO: write best model save, count from best value

In [ ]:
# make the experiment as deterministic as possible
tf.reset_default_graph()
tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)

### *Fashion MNIST* klasės

In [ ]:
classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
classes = pd.DataFrame(
          columns=["Class", "Index"],
          data=zip(classes, range(len(classes))))
classes

,Class,Index
0,T-shirt/top,0
1,Trouser,1
2,Pullover,2
3,Dress,3
4,Coat,4
5,Sandal,5
6,Shirt,6
7,Sneaker,7
8,Bag,8
9,Ankle boot,9


In [ ]:
classes.to_latex(prefix_dir("classes.tex"), index=False)

###Eksperimento parametrų generavimas

In [ ]:
# options to randomly pick from for finding the best parameters
available_params = {
    "Activation function" : ["sigmoid", "tanh", "softmax"],
    "Epoch count" : [10, 30, 50, 70],
    "Learning rate" : [0.01, 0.02, 0.03, 0.04],
    "Batch size" : [32, 64, 128, 256],
    "Kernel size" : [3, 4, 5, 6],
    "Pool size" : [2, 3, 5, 6],
    "Filter count" : [28, 86, 128, 256],
    "Dropout rate" : [0.2, 0.4, 0.5, 0.6],
    "Optimizer" : ["adam", "rmsprop", "adagrad", "sgd"],
    "Architecture": ["full", "no batch norm", "no dropout"]
}

# available_params = {
#     "Activation function" : ["sigmoid", "tanh", "softmax"],
#     "Epoch count" : [10, 30],
#     "Learning rate" : [0.01, 0.02],
#     "Batch size" : [32, 64],
#     "Kernel size" : [3, 4],
#     "Pool size" : [2, 3],
#     "Filter count" : [48, 96],
#     "Dropout rate" : [0.2, 0.4],
#     "Optimizer" : ["adam", "rmsprop"],
#     "Architecture": ["full", "no batch norm"]
# }

# available_params = {
#     "Activation function" : ["sigmoid", "tanh", "softmax"],
#     "Epoch count" : [10],
#     "Learning rate" : [0.01],
#     "Batch size" : [32],
#     "Kernel size" : [3],
#     "Pool size" : [2],
#     "Filter count" : [48],
#     "Dropout rate" : [0.2],
#     "Optimizer" : ["adam"],
#     "Architecture": ["full"]
# }

# sort values
for column in available_params.keys():
  available_params[column] = sorted(available_params[column])

def get_first_params(available_params):
  rand_params = {}
  for column, available in available_params.items():
    rand_params[column] = [available[0]]
  return rand_params

In [ ]:
# set base for experiment
rand_params_base = pd.DataFrame(get_first_params(available_params))

# the params that will be generated from base
params = pd.DataFrame(columns=available_params.keys())

# make a copy of available params because it will be modified
available_params_copy = copy.deepcopy(available_params)

# remove used values from available params copy
for column in rand_params_base.columns:
  available_params_copy[column].remove(rand_params_base.loc[0, column])

# apply available params for every column on base case to params
for column, available in available_params_copy.items():
  while len(available) > 0:
    append_index = params.shape[0]
    # copy last to new row
    params.loc[append_index] = rand_params_base.loc[0]
    # print(params.loc[append_index])
    random_available = available[0]#np.random.choice(available)
    params.loc[append_index, column] = random_available
    available.remove(random_available)

# params
def sort_columns_by_range(available_params, params):
  # available_params_iter = iter(available_params.keys())
  # # ignore first column
  # # don't sort base case
  # column = next(available_params_iter)
  # # [[0]] - returns DataFrame instead of Series
  sorted_series = []
  # sorted_series = [
  #     params.iloc[[0]],
  #     params.iloc[1:len(available_params[column])-1].sort_values(by=column)
  # ]
  # print(sorted_series[0])
  # print(sorted_series[1])
  i = 0
  for column in available_params.keys():
    next_i = i + len(available_params[column])-1
    # print(params.iloc[i:next_i])
    # print(params.iloc[i:next_i].sort_values(by=column))
    # return
    sorted_series.append(params.iloc[i:next_i].sort_values(by=column))
    i = next_i
  return pd.concat(sorted_series)

params = sort_columns_by_range(available_params, params)

# append base case to generated params
params.loc[params.shape[0]] = rand_params_base.loc[0]
params

,Activation function,Epoch count,Learning rate,Batch size,Kernel size,Pool size,Filter count,Dropout rate,Optimizer,Architecture
0,softmax,10,0.01,32,3,2,28,0.2,adagrad,full
1,tanh,10,0.01,32,3,2,28,0.2,adagrad,full
2,sigmoid,30,0.01,32,3,2,28,0.2,adagrad,full
3,sigmoid,50,0.01,32,3,2,28,0.2,adagrad,full
4,sigmoid,70,0.01,32,3,2,28,0.2,adagrad,full
5,sigmoid,10,0.02,32,3,2,28,0.2,adagrad,full
6,sigmoid,10,0.03,32,3,2,28,0.2,adagrad,full
7,sigmoid,10,0.04,32,3,2,28,0.2,adagrad,full
8,sigmoid,10,0.01,64,3,2,28,0.2,adagrad,full
9,sigmoid,10,0.01,128,3,2,28,0.2,adagrad,full


### Papildomos funkcijos palengvinimui

In [ ]:
# do the same func operation inside list elements as if it was a flattened list
mapin = lambda func: lambda lst: type(lst)(map(lambda x:
    func(x) if not isinstance(x, (list, tuple)) else mapin(func)(x), lst))

### Atsisiunčiami *Fashion MNIST* duomenys

In [ ]:
fashion_mnist = datasets.fashion_mnist.load_data()

### Tikrinama duomenų struktūrą

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist
mapin(lambda x: x.shape)(fashion_mnist)

(((60000, 28, 28), (60000,)), ((10000, 28, 28), (10000,)))

### Pavaizduojami antros ir ketvirtos duomenų eilutės paveikslėliai bei tikrosios klasės indeksai

In [ ]:
print(x_train.__class__)
display(x_train[1], x_train[3])
print("klases: ", y_train[1], y_train[3])

<class 'numpy.ndarray'>


array([[  0,   0,   0,   0,   0,   1,   0,   0,   0,   0,  41, 188, 103,
         54,  48,  43,  87, 168, 133,  16,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   1,   0,   0,   0,  49, 136, 219, 216, 228, 236,
        255, 255, 255, 255, 217, 215, 254, 231, 160,  45,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,  14, 176, 222, 224, 212, 203, 198, 196,
        200, 215, 204, 202, 201, 201, 201, 209, 218, 224, 164,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0, 188, 219, 200, 198, 202, 198, 199, 199,
        201, 196, 198, 198, 200, 200, 200, 200, 201, 200, 225,  41,   0,
          0,   0],
       [  0,   0,   0,   0,  51, 219, 199, 203, 203, 212, 238, 248, 250,
        245, 249, 246, 247, 252, 248, 235, 207, 203, 203, 222, 140,   0,
          0,   0],
       [  0,   0,   0,   0, 116, 226, 206, 204, 207, 204, 101,  75,  47,
         73,  48,  50,  45,  51,  63, 113, 222, 202, 206, 220, 224,   0,
          0,   0],
       [  0,   0,   0,   0, 200, 222, 209, 203, 215, 200,   0,  70,  98,
          0, 103,  59,  68,  71,  49,   0, 219, 206, 214, 210, 250,  38,
          0,   0],
       [  0,   0,   0,   0, 247, 218, 212, 210, 215, 214,   0, 254, 243,
        139, 255, 174, 251, 255, 205,   0, 215, 217, 214, 208, 220,  95,
          0,   0],
       [  0,   0,   0,  45, 226, 214, 214, 215, 224, 205,   0,  42,  35,
         60,  16,  17,  12,  13,  70,   0, 189, 216, 212, 206, 212, 156,
          0,   0],
       [  0,   0,   0, 164, 235, 214, 211, 220, 216, 201,  52,  71,  89,
         94,  83,  78,  70,  76,  92,  87, 206, 207, 222, 213, 219, 208,
          0,   0],
       [  0,   0,   0, 106, 187, 223, 237, 248, 211, 198, 252, 250, 248,
        245, 248, 252, 253, 250, 252, 239, 201, 212, 225, 215, 193, 113,
          0,   0],
       [  0,   0,   0,   0,   0,  17,  54, 159, 222, 193, 208, 192, 197,
        200, 200, 200, 200, 201, 203, 195, 210, 165,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,  47, 225, 192, 214, 203, 206,
        204, 204, 205, 206, 204, 212, 197, 218, 107,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   1,   6,   0,  46, 212, 195, 212, 202, 206,
        205, 204, 205, 206, 204, 212, 200, 218,  91,   0,   3,   1,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,  11, 197, 199, 205, 202, 205,
        206, 204, 205, 207, 204, 205, 205, 218,  77,   0,   5,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   3,   0,   2, 191, 198, 201, 205, 206,
        205, 205, 206, 209, 206, 199, 209, 219,  74,   0,   5,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   2,   0,   0, 188, 197, 200, 207, 207,
        204, 207, 207, 210, 208, 198, 207, 221,  72,   0,   4,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   2,   0,   0, 215, 198, 203, 206, 208,
        205, 207, 207, 210, 208, 200, 202, 222,  75,   0,   4,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,   0, 212, 198, 209, 206, 209,
        206, 208, 207, 211, 206, 205, 198, 221,  80,   0,   3,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,   0, 204, 201, 205, 208, 207,
        205, 211, 205, 210, 210, 209, 195, 221,  96,   0,   3,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,   0, 202, 201, 205, 209, 207,
        205, 213, 206, 210, 209, 210, 194, 217, 105,   0,   2,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,   0, 204, 204, 205, 208, 207,
        205, 215, 207, 210, 208, 211, 193, 213, 115,   0,   2,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0, 204, 207, 207, 208, 206,
        206, 215, 210, 210, 207, 212, 195, 210, 118,   0,   2,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   0,   0, 198, 208, 208, 208, 204,
        207, 212, 212, 210, 207, 211, 196, 207, 121,   0,   1,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   1,   

array([[  0,   0,   0,   0,   0,   0,   0,   0,  33,  96, 175, 156,  64,
         14,  54, 137, 204, 194, 102,   0,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,  73, 186, 177, 183, 175, 188, 232,
        255, 223, 219, 194, 179, 186, 213, 146,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,  35, 163, 140, 150, 152, 150, 146, 175,
        175, 173, 171, 156, 152, 148, 129, 156, 140,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0, 150, 142, 140, 152, 160, 156, 146, 142,
        127, 135, 133, 140, 140, 137, 133, 125, 169,  75,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,  54, 167, 146, 129, 142, 137, 137, 131,
        148, 148, 133, 131, 131, 131, 125, 140, 140,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0, 110, 188, 133, 146, 152, 133, 125,
        127, 119, 129, 133, 119, 140, 131, 150,  14,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0, 221, 158, 137, 135, 123, 110,
        110, 114, 108, 112, 117, 127, 142,  77,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   4,   0,  25, 158, 137, 125, 119, 119,
        110, 117, 117, 110, 119, 127, 144,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0, 123, 156, 129, 112, 110,
        102, 112, 100, 121, 117, 129, 114,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0, 125, 169, 127, 119, 106,
        108, 104,  94, 121, 114, 129,  91,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   2,   0,  98, 171, 129, 112, 104,
        114, 106, 102, 112, 104, 133,  64,   0,   4,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   2,   0,  66, 173, 135, 129,  98,
        100, 119, 102, 108,  98, 135,  60,   0,   4,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   2,   0,  56, 171, 135, 127, 100,
        108, 117,  85, 106, 110, 135,  66,   0,   4,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  52, 150, 129, 110, 100,
         91, 102,  94,  83, 104, 123,  66,   0,   4,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   2,   0,  66, 167, 140, 148, 148,
        127, 137, 152, 146, 146, 148,  96,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0,  45, 123,  94, 104,  96,
        119, 121, 106,  98, 112,  87, 114,   0,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0,   0, 106,  89,  58,  50,  37,
         50,  66,  56,  50,  75,  75, 137,  22,   0,   2,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   2,   0,  29, 148, 114, 106, 125,  89,
        100, 133, 117, 131, 131, 131, 125, 112,   0,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0, 100, 106, 114,  91, 137,  62,
        102, 131,  89, 135, 112, 131, 108, 135,  37,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,   0, 146, 100, 108,  98, 144,  62,
        106, 131,  87, 133, 104, 160, 117, 121,  68,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,  33, 121, 108,  96, 100, 140,  71,
        106, 127,  85, 140, 104, 150, 140, 114,  89,   0,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,  62, 119, 112, 102, 110, 137,  75,
        106, 144,  81, 144, 108, 117, 154, 117, 104,  18,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,  66, 121, 102, 112, 117, 131,  73,
        104, 156,  77, 137, 135,  83, 179, 129, 121,  35,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0,  85, 127,  81, 125, 133, 119,  79,
        100, 169,  83, 129, 175,  60, 163, 135, 146,  39,   0,   0,   0,
          0,   0],
       [  0,   0,   0,   0,   0,   0, 10

klases:  0 3


###Normalizuojamos pikselių reikšmes, kad jos nebūtų didelės

In [ ]:
# normalize pixel value range [0, 255] -> [0, 1]
normalized_mnist_dataset = mapin(lambda x: x/255)(fashion_mnist)

###Testavimo ir validavimo perskirstymai sujungiami atgal į aibes

In [ ]:
x = np.concatenate((x_train, x_test))
y = np.concatenate((y_train, y_test))

In [ ]:
x.shape, y.shape

((70000, 28, 28), (70000,))

###Duomenys perskirstomi pagal naują funkciją į testavimo, validavimo ir treniravimo aibes

In [ ]:
def split(x, y, train_percent, valid_percent):
  train_size = int(x.shape[0] * train_percent)
  valid_size = int(x.shape[0] * valid_percent)
  train_slice = slice(0, train_size)
  valid_slice = slice(train_size, train_size + valid_size)
  test_slice = slice(train_size + valid_size, x.shape[0])
  x_train = x[train_slice]
  x_valid = x[valid_slice]
  x_test = x[test_slice]
  y_train = y[train_slice]
  y_valid = y[valid_slice]
  y_test = y[test_slice]
  return (x_train, x_valid, x_test), (y_train, y_valid, y_test)

split_vals = split(x, y, train_percent = 0.8, valid_percent = 0.1)
print(mapin(lambda x: x.shape)(split_vals))
(x_train, x_valid, x_test), (y_train, y_valid, y_test) = split_vals

(((56000, 28, 28), (7000, 28, 28), (7000, 28, 28)), ((56000,), (7000,), (7000,)))


In [ ]:
x_train.__class__

numpy.ndarray

###Sukuriamas neuroninis tinklas

In [ ]:
# https://www.tensorflow.org/tutorials/images/cnn

class ConvNet():
  def __init__(self, architecture_name, activation, dropout_rate, pool_size, kernel_size, filter_count):
    self.architect(architecture_name, dropout_rate, pool_size, kernel_size, filter_count)

  def architect(self, architecture_name, activation, dropout_rate, pool_size, kernel_size, filter_count):
    model = models.Sequential()
    model.add(layers.Conv2D(filter_count, (kernel_size, kernel_size), activation='relu', input_shape=(28, 28, 1)))
    model.add(layers.MaxPooling2D((pool_size, pool_size)))
    model.add(layers.Dropout(dropout_rate)) if architecture_name != 'no dropout' else None
    model.add(layers.BatchNormalization()) if architecture_name != 'no batch norm' else None
    model.add(layers.Conv2D(2*filter_count, (kernel_size, kernel_size), activation='relu'))
    model.add(layers.MaxPooling2D((pool_size, pool_size)))
    model.add(layers.Dropout(dropout_rate)) if architecture_name != 'no dropout' else None
    model.add(layers.BatchNormalization()) if architecture_name != 'no batch norm' else None
    model.add(layers.Conv2D(2*filter_count, (kernel_size, kernel_size), activation='relu'))
    model.add(layers.BatchNormalization()) if architecture_name != 'no batch norm' else None
    model.add(layers.Flatten())
    model.add(layers.Dense(56, activation='relu'))
    model.add(layers.BatchNormalization()) if architecture_name != 'no batch norm' else None
    model.add(layers.Dense(10, activation=activation))
    self.model = model

  def compile(self, optimizer, metrics=['accuracy']):
    self.model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=metrics)

  def train(self, x, y, x_valid, y_valid, batch_size, epochs):
    return self.model.fit(
        x, y, epochs=epochs,
        validation_data=(x_valid, y_valid),
        batch_size=batch_size
      )

  def evaluate(self, x, y, callbacks):
    loss, accuracy = self.model.evaluate(x, y, callbacks=callbacks, verbose=2)
    return loss, accuracy * 100

  def get_predictions(self, x):
    return self.model.predict(x)

  def summarize(self):
    self.model.summary()

In [ ]:
a = np.arange(10, 50, 5)
print(a)
b = np.arange(20, 60, 5)
b = np.append(b, 25)
print(b)
c = [3]
c.extend(np.arange(10))
c.extend(np.arange(10))
print(c)

[10 15 20 25 30 35 40 45]
[20 25 30 35 40 45 50 55 25]
[3, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


###Modelių sudarymo ciklas su skirtingais parametrais

In [ ]:
import pandas as pd

data = {
  "name": ["Sally", "Mary", "John"],
  "age": [50, 40, 30],
  "qualified": [True, False, False]
}

df = pd.DataFrame(data)

newdf = df.drop(1, axis='index')

print(newdf)

    name  age  qualified
0  Sally   50       True
2   John   30      False


In [ ]:
import pandas as pd

data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'], 'Age': [25, 30, 28, 35]}
df = pd.DataFrame(data, index=['a', 'b', 'c', 'd'])
df.drop(df.iloc[2].name)

,Name,Age
a,Alice,25
b,Bob,30
d,David,35


In [ ]:
a = [1,2,3]
a.remove(3)
a

[1, 2]

In [ ]:
# https://colab.research.google.com/notebooks/gpu.ipynb#scrollTo=sXnDmXR7RDr2

results = {"Loss": [], "Accuracy": []}
# dummy learning rate initialization for updating in the loop
# https://medium.com/@danielonugha0/how-to-change-the-learning-rate-of-tensorflow-b5d854819050
learning_rate = tf.Variable(0.01, trainable=False)
max_accuracy, min_loss = 0, np.inf
best_predictions = None
LOSS_BASELINE = 0.2
ACCURACY_BASELINE = 80
device = '/device:GPU:0' if tf.test.gpu_device_name() == '/device:GPU:0' else '/cpu:0'

for i in range(len(params)):
  print(f"\nPARAM_SET: {i}")
  row = params.iloc[i]
  print(row)
  try:
    model = ConvNet(row.loc["Architecture"], row.loc["Activation"], row.loc["Dropout rate"], row.loc["Pool size"], row.loc["Kernel size"], row.loc["Filter count"])
    model.summarize()
    model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
      filepath=prefix_dir("saved_best_acc.model.keras"),
      monitor='accuracy',
      mode='max',
      save_best_only=True)
    model.compile(row.loc["Optimizer"])
    tf.keras.backend.set_value(learning_rate, row.loc["Learning rate"])
    # x_test = np.array(x_test)
    # x_valid = np.array(x_valid)
    x_full_test = np.concatenate((x_valid, x_test))
    y_full_test = np.concatenate((y_valid, y_test))
    with tf.device(device):
      history = model.train(x_train, y_train, x_valid, y_valid, row.loc["Batch size"], row.loc["Epoch count"])
      loss, accuracy = model.evaluate(x_full_test, y_full_test, callbacks=[model_checkpoint_callback])
      loss = round((loss - LOSS_BASELINE) / LOSS_BASELINE * 100, 2)
      accuracy = round((accuracy - ACCURACY_BASELINE) / ACCURACY_BASELINE * 100, 2)
      if accuracy > max_accuracy:
        max_accuracy = accuracy
        min_loss = loss
        best_predictions = model.get_predictions(x_full_test)
    results["Loss"].append(loss)
    results["Accuracy"].append(accuracy)
    print(f"\nLoss: {loss:.2f}%, Accuracy: {accuracy:.2f}%\n")
  except:
    print("\n\n-------------Exception occurred, continuing...-------------\n\n")
    # remove these params from experiment
    params = params.drop(params.iloc[i].name)
    for column, value in available_params.items():
      available_params[column].remove(value)
    continue

PARAMS_LOSS_NAME = f"Loss percent change from baseline {LOSS_BASELINE}"
PARAMS_ACC_NAME = f"Accuracy percent change from baseline {ACCURACY_BASELINE}%"

params["Loss"] = results["Loss"]
params["Accuracy"] = results["Accuracy"]


PARAM_SET: 0
Activation function    softmax
Epoch count                 10
Learning rate             0.01
Batch size                  32
Kernel size                  3
Pool size                    2
Filter count                28
Dropout rate               0.2
Optimizer              adagrad
Architecture              full
Name: 0, dtype: object
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_9 (Conv2D)           (None, 26, 26, 28)        280       
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 13, 13, 28)        0         
 g2D)                                                            
                                                                 
 dropout_6 (Dropout)         (None, 13, 13, 28)        0         
                                                                 
 batch_normalization_12 (Ba  (None, 1

In [ ]:
params.to_csv(prefix_dir("params.csv"), index=False)
params

In [ ]:
print(f"Min loss from baseline: {min_loss:.2f}%, Max accuracy from baseline: {max_accuracy:.2f}%")

###Kiekvieno modelio klaidų ir tikslumo grafikas parametrų atžvilgiu.

In [ ]:
# https://matplotlib.org/stable/gallery/lines_bars_and_markers/barchart.html#sphx-glr-gallery-lines-bars-and-markers-barchart-py

def plot_multi_bars(
    x_labels, y_bar_name_to_data, x_title):
  # data in order to labels

  x = np.arange(len(x_labels))  # the label locations
  width = 0.25  # the width of the bars
  multiplier = 0

  fig, ax = plt.subplots(layout='constrained')

  for attribute, measurements in y_bar_name_to_data.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, measurements, width, label=attribute)
    ax.bar_label(rects, padding=10)
    multiplier += 1

  # Add some text for labels, title and custom x-axis tick labels, etc.
  ax.set_ylabel('Percentage')
  ax.set_xlabel(x_title)
  ax.set_title(f'Loss and accuracy with respect to changed {x_title}')
  ax.set_xticks(x + width, x_labels)
  ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncols=2)
  ax.set_ylim(0, 50)
  create_dir(prefix_dir("acc_loss"))
  plt.savefig(prefix_dir(f'acc_loss/{x_title}.png'))

In [ ]:
# generate just random 2 dataframes inline
def generate_random_dataframe():
  return pd.DataFrame(np.random.randint(1, 100, size=(10, 4)), columns=['A', 'B', 'C', 'D'])

generate_random_dataframe().iloc[3:5][0:3].values.tolist()

In [ ]:
def plot_all_param_results(available_params, params):
  i = 0
  # last rows are not counted as they are accuracy and loss
  last_param_index = len(params) - 1
  for column, values in available_params.items():
    next_i = i + len(values) - 1
    # append last param (base) for comparison
    plot_multi_bars(
      x_labels=params[column][i:next_i].tolist() + [params[column][last_param_index]],
      y_bar_name_to_data={
          PARAMS_LOSS_NAME: params['Loss'][i:next_i].tolist() + [params['Loss'][last_param_index]],
          PARAMS_ACC_NAME: params['Accuracy'][i:next_i].tolist() + [params['Accuracy'][last_param_index]]
      },
      x_title=column
    )
    i = next_i

plot_all_param_results(available_params, params)

###Geriausio modelio 30 spėjimų

In [ ]:
best_predictions = best_predictions.argmax(axis=1)
prediction_results = {"Actual index": [], "Predicted": [], "Actual": []}
for i_class in classes["Index"]:
  class_actual_indices = np.where(y_full_test == i_class)[0][0:3]
  prediction_results["Actual index"].extend(class_actual_indices)
  prediction_results["Predicted"].extend(best_predictions[class_actual_indices])
  prediction_results["Actual"].extend([i_class] * 3)
prediction_results = pd.DataFrame(prediction_results)
prediction_results

In [ ]:
prediction_results.to_latex(prefix_dir("results.tex"), index=False)

In [ ]:
# plt.plot(history.history['accuracy'], label='accuracy')
# plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.ylim([0.5, 1])
# plt.legend(loc='lower right')

###Geriausio modelio *confusion* matrica

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html
confusion_matrix_obj = confusion_matrix(y_full_test, best_predictions, labels=classes["Index"])
confusion_matrix_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix_obj, display_labels=classes["Index"])
# important for plt.savefig before plt.show
confusion_matrix_display.plot().figure_.savefig(prefix_dir("confusion_matrix.png"))
# confusion_matrix_display.plot()